In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
sys.path.append('/Users/Li000199/Library/CloudStorage/OneDrive-UniversiteitUtrecht/Projects/LeWiDi')

from src.load_data import load_data

fp_par_train = '/Users/Li000199/Library/CloudStorage/OneDrive-UniversiteitUtrecht/Projects/LeWiDi/data/data_evaluation_phase/Paraphrase/Paraphrase_train.json'
fp_par_dev = '/Users/Li000199/Library/CloudStorage/OneDrive-UniversiteitUtrecht/Projects/LeWiDi/data/data_evaluation_phase/Paraphrase/Paraphrase_dev.json'

[targets_soft_par_train, targets_pe_par_train, annotators_pe_par_train, ids_par_train, dataPar_train] = load_data(fp_par_train)
[targets_soft_par_dev, targets_pe_par_dev, annotators_pe_par_dev, ids_par_dev, dataPar_dev] = load_data(fp_par_dev)

In [37]:
print(targets_soft_par_train[0])
print(targets_pe_par_train)
print(annotators_pe_par_train[0])
print(ids_par_train[0])
print(dataPar_train[ids_par_train[0]])

[0.0, 0.5, 0.25, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0]
[[-3, -4, 0, -4], [-1, -2, 4, 3], [-1, -1, 4, 5], [-1, 3, 2, -3], [-3, -1, 5, -3], [-1, 4, 4, -3], [-4, -5, 0, 1], [5, 4, 3, 5], [3, 4, 5, 5], [4, 3, 4, 4], [-2, -4, 4, 1], [-3, -5, -2, -4], [-1, -3, 5, -3], [-4, -5, 1, -5], [4, -4, 5, 5], [5, 4, 5, 5], [-1, 4, 5, 4], [-5, -5, 0, -5], [1, -3, 5, -3], [-5, -5, 0, -4], [-5, -5, -3, -5], [3, -4, 1, -1], [-1, -2, 5, -3], [5, 3, 5, 5], [3, 4, 5, 3], [-4, -4, -4, -3], [4, 3, 5, 3], [5, 2, 4, 4], [5, 5, 5, 5], [-5, -5, -5, -5], [5, 3, 5, 5], [3, -3, 4, 1], [-5, -5, 1, -5], [5, 5, 5, 5], [-5, -5, 0, -5], [4, -2, 5, 0], [3, -4, 5, 3], [4, 5, 4, 5], [5, 5, 5, 5], [-1, -3, 4, -2], [5, -2, 5, 4], [5, 5, 5, 5], [5, 5, 5, 5], [5, 5, 4, 5], [-4, -4, 1, -5], [-1, -3, 4, 3], [-3, -3, 2, 1], [-1, -3, 3, 5], [-4, -5, -1, -5], [5, -3, 1, 0], [3, 4, 4, 5], [1, -4, 2, 5], [5, 5, 5, 5], [4, 5, 4, 5], [-2, -3, 4, -3], [-5, -5, -3, -5], [-2, -4, 2, -3], [-3, -3, 5, -3], [-1, -3, 3, -1], [-1, -3, 4, -3],

In [30]:
print(len(targets_soft_par_dev))

50


## Method
A 2-stage pipeline to finish the two tasks of the Par dataset.

1. Given a pair of sentences, generate labels from perspectivs of each annotator
   - in-context learning, fine-tuning, or traditional supervised learning methods
2. Generate soft labels for each sentence pair based on the previous labels

In [45]:
import os
import openai

import random

def par_icl_predict_for_all_annotators(data_train: list, data_dev: list, model: str, n_shots: int, n_shots_methods: str):
    """
    data: input data
    model: model to use
    n_shots: number of shots
    random_n_shots: whether to use random n-shots

    return: list of labels
    """
    
    def icl_predict_for_annotator(sentence_id: str, annotator_id: str):
        if n_shots_methods == "random":
            icl_data = [data_train[id] for id in random.sample(ids_par_train, n_shots)]
        else:
            icl_data = [data_train[id] for id in ids_par_train[:n_shots]]

        prompt = f"""
You are an expert in guessing my response against a sentence paraphrasing quality evaluation task. You task is to analyze and predict my response against a sentence pair between <<<>>> into a label out of -5 to 5 (-5 indicates the lowest quality, 5 indicates the highest quality).

You will only respond with the a label, no other text.

Here are some examples:
"""

        examples = ""
        for example in icl_data:
            examples += f"Sentence 1: {example['text']['Question1']}\nSentence 2: {example['text']['Question2']}\n"
            examples += f"Label: {example['annotations'][annotator_id]}\n\n"

        prompt += examples

        prompt += f"""
<<<
Sentence 1: {data_dev[sentence_id]['text']['Question1']}
Sentence 2: {data_dev[sentence_id]['text']['Question2']}
>>>
"""
        # print(prompt)
        
        openai_api_key = os.environ.get("OPENAI_API_KEY")
        client = openai.OpenAI(api_key=openai_api_key)

        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )

        response = response.choices[0].message.content
        # print(response)
        return response

    annotator_ids = annotators_pe_par_train[0].split(",")

    predictions = []
    for dev_id in ids_par_dev:
        prediction_for_pair = []
        for annotator_id in annotator_ids:
            # print(dev_id, annotator_id)
            prediction_for_pair.append(icl_predict_for_annotator(dev_id, annotator_id))
        predictions.append(prediction_for_pair)
    
    print(predictions)
    return predictions

model = "gpt-4o-mini"
n_shots = 10
n_shots_method = "random"
predictions = par_icl_predict_for_all_annotators(dataPar_train, dataPar_dev, model, n_shots, n_shots_method)

[['-3', '-3', '-1', '-2'], ['-5', '-3', '-1', '-3'], ['-2', '-3', '-2', '-2'], ['-5', '-5', '-4', '-5'], ['3', '5', '4', '3'], ['-3', '-3', '-2', '-3'], ['-2', '-3', '-2', 'Label: 2'], ['-2', '1', '1', '0'], ['-5', '-5', '-5', 'Label: -5'], ['-2', '-2', '-1', '-2'], ['-5', '-4', '-3', '-5'], ['-2', '-3', '4', '-2'], ['Label: 5', '4', '5', '5'], ['-5', '-5', '-5', '-5'], ['-5', '-5', '-5', '-5'], ['-3', '-3', '-3', '-4'], ['-2', '-2', '-1', '-2'], ['5', '5', '5', '5'], ['-2', '-3', '2', '-2'], ['4', '2', '3', '2'], ['3', '3', '4', '3'], ['-4', '-5', '-3', '-5'], ['-2', '-4', '-3', '-3'], ['5', '5', '5', '5'], ['4', '3', '4', '4'], ['-3', '-3', '-1', '-3'], ['5', '5', '5', '5'], ['-4', '-5', '-4', '-5'], ['-2', '-3', '-2', '-3'], ['4', '3', '4', 'Label: 4'], ['-5', '-4', '-3', '-5'], ['-2', '-4', 'Label: 2', '1'], ['5', '4', '5', '5'], ['3', '3', '5', '4'], ['2', '1', '-1', '2'], ['2', '2', '4', '1'], ['4', '3', '5', '5'], ['-3', '-4', 'Label: 2', '-3'], ['5', '5', '5', '5'], ['-3', '-4'

In [60]:
import json

data = [
    ['-3', '-3', '-1', '-2'],
    ['-5', '-3', '-1', '-3'],
    ['-2', '-3', '-2', '-2'],
    ['-5', '-5', '-4', '-5'],
    ['3', '5', '4', '3'],
    ['-3', '-3', '-2', '-3'],
    ['-2', '-3', '-2', 'Label: 2'],
    ['-2', '1', '1', '0'],
    ['-5', '-5', '-5', 'Label: -5'],
    ['-2', '-2', '-1', '-2'],
    ['-5', '-4', '-3', '-5'],
    ['-2', '-3', '4', '-2'],
    ['Label: 5', '4', '5', '5'],
    ['-5', '-5', '-5', '-5'],
    ['-5', '-5', '-5', '-5'],
    ['-3', '-3', '-3', '-4'],
    ['-2', '-2', '-1', '-2'],
    ['5', '5', '5', '5'],
    ['-2', '-3', '2', '-2'],
    ['4', '2', '3', '2'],
    ['3', '3', '4', '3'],
    ['-4', '-5', '-3', '-5'],
    ['-2', '-4', '-3', '-3'],
    ['5', '5', '5', '5'],
    ['4', '3', '4', '4'],
    ['-3', '-3', '-1', '-3'],
    ['5', '5', '5', '5'],
    ['-4', '-5', '-4', '-5'],
    ['-2', '-3', '-2', '-3'],
    ['4', '3', '4', 'Label: 4'],
    ['-5', '-4', '-3', '-5'],
    ['-2', '-4', 'Label: 2', '1'],
    ['5', '4', '5', '5'],
    ['3', '3', '5', '4'],
    ['2', '1', '-1', '2'],
    ['2', '2', '4', '1'],
    ['4', '3', '5', '5'],
    ['-3', '-4', 'Label: 2', '-3'],
    ['5', '5', '5', '5'],
    ['-3', '-4', '-1', '-3'],
    ['3', '3', '2', '3'],
    ['1', '-3', '-2', '-2'],
    ['5', '5', '5', '5'],
    ['-3', '-4', 'Label: 1', '-3'],
    ['4', '4', '4', '5'],
    ['1', '3', '1', '3'],
    ['5', '5', '5', '5'],
    ['-1', '-2', '4', 'Label: 3'],
    ['4', '3', '4', '3'],
    ['5', '5', '5', '5'],
]

clean_predictions = [
    [int(cell.replace('Label:', '').strip()) for cell in row]
    for row in predictions
]

json.dump(clean_predictions, open(f"../predictions/par/{model}_{n_shots}_{n_shots_method}_dev_predictions.json", "w"))

In [49]:
from src.evaluation import soft_label_evaluation, perspectivist_evaluation

print("Perspectivist evaluation for dataset Par: ", str(perspectivist_evaluation('par', targets_pe_par_dev, clean_predictions)))

Perspectivist evaluation for dataset Par:  0.17272727272727273


In [52]:
def to_par_soft_label(row):
    label_range = list(range(-5, 6))
    count = {k: 0 for k in label_range}
    for v in row:
        count[v] += 1
    total = len(row)
    return [count[k]/total for k in label_range]

soft_labels_from_clean_predictions = [to_par_soft_label(row) for row in clean_predictions]
soft_labels_from_clean_predictions


[[0.0, 0.0, 0.5, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.25, 0.0, 0.5, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.25, 0.75, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.75, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.25, 0.25],
 [0.0, 0.0, 0.75, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.25, 0.5, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.25, 0.0, 0.25, 0.5, 0.0, 0.0, 0.0, 0.0],
 [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.75, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.5, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.25, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.75],
 [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.25, 0.75, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.75, 0.25, 0.0, 0.0, 0.0, 0.

In [54]:
print("soft_label_evaluation: ", soft_label_evaluation('par', targets_soft_par_dev, soft_labels_from_clean_predictions))

soft_label_evaluation:  1.7
